# 🧬 CDK4 Inhibitor Design Pipeline

Copyright (c) 2026, NVIDIA CORPORATION. Licensed under the Apache License, Version 2.0 (the "License") you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

---

This notebook provides an end-to-end workflow for designing **CDK4-selective inhibitors** while avoiding **CDK11** binding.

## Pipeline Overview

1. **Setup & Configuration** - Install dependencies and configure the environment
2. **NIM Service Check** - Verify MolMIM and Boltz2 availability
3. **Seed Definition** - Specify starting molecules
4. **Molecule Generation** - Generate novel candidates with MolMIM
5. **Affinity Prediction** - Predict CDK4/CDK11 binding with Boltz2
6. **Physicochemical Analysis** - Calculate drug-likeness properties
7. **Composite Scoring** - Rank compounds by multi-objective score
8. **Visualization & Report** - Generate results and plots
9. **Synthesis Pathway Prediction** - Predict synthesizable routes with ReaSyn

---

## 1. Setup & Configuration

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path, whether the notebook is launched from repo root
# or from the mini-hands-on/ directory.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / "cdk_oracle").exists() else CWD.parent
sys.path.insert(0, str(PROJECT_ROOT))
MINI_HANDS_ON_DIR = PROJECT_ROOT / "mini-hands-on"

# Import pipeline components
from cdk_oracle import (
    CDKConfig,
    NIMHealthChecker,
    MolMIMClient,
    Boltz2AffinityClient,
    PhysicochemCalculator,
    CDKScorer,
    CDKVisualizer,
    CDKDesignPipeline
)

print("✓ CDK Oracle package loaded")
print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Mini hands-on directory: {MINI_HANDS_ON_DIR}")


In [ ]:
# Initialize configuration
# You can modify these settings directly or via environment variables

# Demo mode is tuned for a live workshop on one Boltz-2 endpoint.
# Set OPENHACKATHON_DEMO_MODE=0 for a larger exploratory run.
DEMO_MODE = os.environ.get("OPENHACKATHON_DEMO_MODE", "1").lower() not in {"0", "false", "no"}
CMA_ITERATIONS = int(os.environ.get("OPENHACKATHON_CMA_ITERATIONS", "2" if DEMO_MODE else "10"))
CMA_POPSIZE = int(os.environ.get("OPENHACKATHON_CMA_POPSIZE", "8" if DEMO_MODE else "20"))
TOP_K_FOR_BOLTZ2 = int(os.environ.get("OPENHACKATHON_TOP_K_FOR_BOLTZ2", "2" if DEMO_MODE else "10"))
TOP_N_COMPOUNDS = int(os.environ.get("OPENHACKATHON_TOP_N_COMPOUNDS", "5" if DEMO_MODE else "20"))
SAVE_BOLTZ2_STRUCTURES = os.environ.get(
    "OPENHACKATHON_SAVE_STRUCTURES", "0" if DEMO_MODE else "1"
).lower() in {"1", "true", "yes"}

config = CDKConfig(
    # NIM URLs are loaded from .openhackathon-nims.env when present, or from MOLMIM_URL/BOLTZ2_URL.
    boltz2_url=os.environ.get("BOLTZ2_URL", "http://localhost:8000"),
    molmim_url=os.environ.get("MOLMIM_URL", "http://localhost:8001"),
    
    # Optimization parameters
    cma_iterations=CMA_ITERATIONS,  # Number of CMA-ES iterations
    cma_popsize=CMA_POPSIZE,     # Population size per seed
    cma_sigma=1.0,          # CMA-ES step size
    
    # Scoring weights (must sum to 1.0)
    weights={
        "binding_affinity": 0.25,   # CDK4 binding potency
        "selectivity": 0.20,        # CDK11/CDK4 ratio
        "cdk11_avoidance": 0.15,    # Penalty for CDK11 binding
        "qed": 0.15,                # Drug-likeness
        "sa": 0.10,                 # Synthetic accessibility
        "pains": 0.10,              # PAINS filter
        "novelty": 0.05             # Structural novelty
    },
    
    # Output settings - timestamped folder created automatically!
    # Each run saves to: output/run_YYYYMMDD_HHMMSS/
    # To use a fixed folder: output_dir=Path("./my_output"),
    output_base=MINI_HANDS_ON_DIR / "output",  # Base folder for timestamped mini hands-on runs
    top_n_compounds=TOP_N_COMPOUNDS,
    save_boltz2_structures=SAVE_BOLTZ2_STRUCTURES,
)

# Optional multi-endpoint mode. For Apptainer/Singularity deployments, set e.g.:
# export BOLTZ2_ENDPOINTS="http://localhost:8010,http://localhost:8011"
endpoint_env = os.environ.get("BOLTZ2_ENDPOINTS", "").strip()
BOLTZ2_ENDPOINTS = [
    {"url": url.strip()} for url in endpoint_env.split(",") if url.strip()
] or None

print("Configuration:")
print(f"  Mode: {'demo' if DEMO_MODE else 'full'}")
print(f"  Target: {config.on_target}")
print(f"  Anti-target: {config.anti_target}")
print(f"  MolMIM URL: {config.molmim_url}")
print(f"  Boltz2 URL: {config.boltz2_url}")
print(f"  Extra Boltz2 endpoints: {len(BOLTZ2_ENDPOINTS) if BOLTZ2_ENDPOINTS else 0}")
print(f"  CMA iterations: {config.cma_iterations}")
print(f"  CMA population: {config.cma_popsize}")
print(f"  Top-K for Boltz2: {TOP_K_FOR_BOLTZ2}")
print(f"  Save Boltz2 structures: {config.save_boltz2_structures}")
print(f"  Output directory: {config.output_dir}")


## 2. NIM Service Health Check

In [ ]:
# Check NIM services availability (passes extra Boltz2 endpoints if configured)
health_checker = NIMHealthChecker(config, boltz2_endpoints=BOLTZ2_ENDPOINTS)
service_status = health_checker.print_status()

# Store availability flags
MOLMIM_AVAILABLE = service_status.get("molmim", False)
BOLTZ2_AVAILABLE = service_status.get("boltz2", False)

if not BOLTZ2_AVAILABLE:
    print("\n⚠️  Boltz2 not available - affinity predictions will fail")
    print("   Start Boltz2 NIM or set BOLTZ2_URL to a running instance")

if not MOLMIM_AVAILABLE:
    print("\n⚠️  MolMIM not available - molecule generation will fail")
    print("   Start MolMIM NIM or set MOLMIM_URL to a running instance")

## 3. Define Seed Molecules

Start with known CDK4 inhibitors or your own designs.

In [ ]:
# Define seed molecules for optimization
# These are CORRECT SMILES for FDA-approved CDK4/6 inhibitors

SEED_MOLECULES = [
    # Palbociclib (Ibrance) - FDA-approved CDK4/6 inhibitor
    # CHEMBL189963, C24H29N7O2, MW=447.5
    "CC(=O)c1c(C)c2cnc(Nc3ccc(N4CCNCC4)cn3)nc2n(C5CCCC5)c1=O",
    
    # Ribociclib (Kisqali) - FDA-approved CDK4/6 inhibitor
    # CHEMBL3545110, correct SMILES from ChEMBL database
    "CN(C)C(=O)c1cc2cnc(Nc3ccc(N4CCNCC4)cn3)nc2n1C1CCCC1",
    
    # Abemaciclib (Verzenio) - FDA-approved CDK4/6 inhibitor
    # CHEMBL3301610, C27H32F2N8, MW=506.6
    # "CCN1CCN(Cc2ccc(Nc3ncc(F)c(-c4cc(F)c5nc(C)n(C(C)C)c5c4)n3)nc2)CC1",
]

if DEMO_MODE:
    SEED_MOLECULES = SEED_MOLECULES[:1]

print(f"Defined {len(SEED_MOLECULES)} seed molecules:")
for i, smi in enumerate(SEED_MOLECULES):
    print(f"  {i+1}. {smi[:60]}...")

In [ ]:
# Visualize seed molecules (optional)
from rdkit import Chem
from rdkit.Chem import Draw

mols = [Chem.MolFromSmiles(smi) for smi in SEED_MOLECULES]
legends = [f"Seed {i+1}" for i in range(len(mols))]

img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 200), legends=legends)
display(img)

## 4. Run Design Pipeline

Execute the complete workflow with **CDK-aware optimization loop**:

### Optimization Loop (per iteration)
```
┌─────────────────────────────────────────────────────────────────────┐
│  CMA-ES generates 20 candidates                                     │
│                          ↓                                          │
│  FAST FILTER (all 20):                                              │
│    • Validity check                                                 │
│    • QED (drug-likeness)                                            │
│    • SA score (synthetic accessibility)                             │
│    • PAINS filter                                                   │
│    • Novelty vs reference                                           │
│                          ↓                                          │
│  SELECT TOP-K (e.g., 10) by fast score                              │
│                          ↓                                          │
│  BOLTZ2 PREDICTIONS (top 10 only):                                  │
│    • CDK4 binding affinity (IC50)                                   │
│    • CDK11 binding affinity (IC50)                                  │
│    • Calculate selectivity ratio                                    │
│                          ↓                                          │
│  COMPOSITE SCORE = weighted sum of:                                 │
│    binding + selectivity + avoidance + qed + sa + pains + novelty   │
│                          ↓                                          │
│  FEEDBACK TO CMA-ES (guides next iteration)                         │
└─────────────────────────────────────────────────────────────────────┘
```

This hybrid approach:
- **Saves time**: Only top candidates get expensive Boltz2 calls
- **Maintains quality**: Full CDK scoring guides optimization
- **Tracks progress**: Best compounds saved throughout loop

In [ ]:
# Initialize pipeline with optional multi-endpoint Boltz2 for parallel predictions
# If BOLTZ2_ENDPOINTS is set (above), predictions run in parallel across all endpoints
pipeline = CDKDesignPipeline(config, boltz2_endpoints=BOLTZ2_ENDPOINTS)

# Run the complete pipeline with CDK-aware optimization
# 
# Per iteration:
#   1. Generate configured popsize candidates via CMA-ES
#   2. Fast filter all candidates on: validity, QED, SA, PAINS, novelty
#   3. Select TOP_K_FOR_BOLTZ2 for expensive Boltz2 predictions
#   4. Predict CDK4 & CDK11 affinities for top-K (parallel if multiple endpoints!)
#   5. Compute full composite score (affinity + selectivity + drug-likeness)
#   6. Feed scores back to CMA-ES for next iteration
#
# After all iterations: final batch Boltz2 scoring on all unique valid molecules

results = pipeline.run(
    seed_smiles=SEED_MOLECULES,
    num_iterations=config.cma_iterations,
    popsize=config.cma_popsize,
    top_k_for_boltz2=TOP_K_FOR_BOLTZ2,
    use_cma=True,                          # Use CMA-ES optimization
    use_msa=True,                          # Use MSA for Boltz2 (better accuracy)
    reference_smiles=SEED_MOLECULES,       # For novelty scoring
    verbose=True,
    generate_report=True,
    save_structures=SAVE_BOLTZ2_STRUCTURES
)

## 5. View Results

In [ ]:
# Best compounds identified DURING optimization (with in-loop Boltz2 scoring)
# These were the top candidates at each iteration that received full CDK affinity predictions
print("="*80)
print("BEST COMPOUNDS FROM OPTIMIZATION LOOP")
print("(These were scored with Boltz2 during CMA-ES iterations)")
print("="*80)

loop_best = pipeline.get_best_compounds_from_generation(n=15)
if len(loop_best) > 0:
    display(loop_best[[
        "smiles", "cdk4_ic50", "cdk11_ic50", "selectivity", 
        "qed", "total_score", "seed_idx", "iteration"
    ]].head(15))
else:
    print("No in-loop Boltz2 predictions available")

In [ ]:
# Final ranked compounds (after complete batch scoring)
print("="*80)
print("TOP 10 COMPOUNDS (FINAL RANKING)")
print("="*80)

top_10 = results.get_top_compounds(10)
display(top_10[[
    "rank", "compound_id", "cdk4_ic50_nm", "cdk11_ic50_nm", 
    "selectivity_ratio", "qed", "total_score"
]])

In [ ]:
# Summary statistics
print("="*80)
print("PIPELINE SUMMARY")
print("="*80)
for key, value in results.summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

# Optimization loop statistics
print("\nOptimization Loop Summary:")
print(f"  Total Boltz2 calls during loop: {len(pipeline._best_compounds) if hasattr(pipeline, '_best_compounds') else 'N/A'}")
if hasattr(pipeline, '_history') and pipeline._history:
    import numpy as np
    boltz2_calls = sum([h.get('boltz2_count', 0) for h in pipeline._history])
    print(f"  Boltz2 predictions in loop: {boltz2_calls}")
    avg_valid = np.mean([h.get('valid_count', 0) for h in pipeline._history])
    print(f"  Avg valid molecules/iteration: {avg_valid:.1f}")

## 6. Save Results

In [ ]:
# Results are AUTO-SAVED by pipeline.run() to a timestamped folder!
# This cell shows what was saved:

print(f"Results saved to: {config.output_dir}")
print("\nFiles automatically generated:")
print("  ✓ all_compounds_scores.csv  (All molecules with full oracle scores)")
print("  ✓ top_compounds.csv         (Top N compounds)")
print("  ✓ run_summary.json          (Pipeline statistics)")
print("  ✓ generated_smiles.txt      (All generated SMILES)")
print("  ✓ seed_molecules.txt        (Input seeds)")
print("  ✓ design_report.html        (Interactive HTML report)")

# Optional: Save additional results manually
# results.save(config.output_dir)

### [optional] Export for Submission

In [ ]:
# Create submission file
submission_df = results.get_top_compounds(25)[["smiles", "compound_id"]].copy()
submission_df.to_csv(config.output_dir / "submission.csv", index=False)

print(f"Submission file created: {config.output_dir / 'submission.csv'}")
print(f"Contains {len(submission_df)} compounds")

---

## 7. Optional Synthesis Pathway Prediction (ReaSyn)

Use [ReaSyn](https://github.com/NVIDIA-Digital-Bio/ReaSyn) to predict **synthesizable routes** for the top-ranked CDK4 inhibitors.

ReaSyn generates retrosynthetic pathways using iterative refinement:
1. Bottom-up autoregressive generation
2. Top-down refinement
3. Holistic editflow refinement

> **Optional prerequisite**: A ReaSyn MCP Server must be running in HTTP mode. Set `REASYN_URL` if it is not on `localhost:8020`. If the server or `fastmcp` client is unavailable, the following cells will skip gracefully.


In [ ]:
import json
import os
import asyncio
import pandas as pd
from IPython.display import display, HTML

try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

REASYN_URL = os.environ.get("REASYN_URL", "http://localhost:8020")
reasyn_results = {}
compound_ids = []
smiles_list = []

try:
    from fastmcp import Client
except ImportError:
    print("fastmcp is not installed; skipping optional ReaSyn synthesis prediction.")
    print("Install it and start a ReaSyn MCP server to enable this section.")
else:
    top_15 = results.get_top_compounds(15)
    smiles_list = top_15["smiles"].tolist()
    compound_ids = top_15["compound_id"].tolist()

    print(f"Submitting {len(smiles_list)} compounds to ReaSyn for synthesis pathway prediction...")
    print(f"ReaSyn endpoint: {REASYN_URL}")
    print(f"Compounds: {compound_ids}")

    async def call_reasyn():
        async with Client(f"{REASYN_URL}/mcp") as client:
            result = await client.call_tool(
                "reconstruct_batch",
                {
                    "smiles_list": smiles_list,
                    "num_cycles": 12,
                    "search_width": 4,
                    "exhaustiveness": 8,
                    "time_limit": 10000,
                },
            )
            return result

    try:
        raw_result = asyncio.get_event_loop().run_until_complete(call_reasyn())
        reasyn_results = json.loads(raw_result.content[0].text) if raw_result and raw_result.content else {}
        print(f"\nReaSyn returned results for {len(reasyn_results.get('results', []))} compounds.")
    except Exception as exc:
        print(f"ReaSyn request failed; skipping optional synthesis analysis: {type(exc).__name__}: {exc}")
        reasyn_results = {}


In [ ]:
if reasyn_results:
    from cdk_oracle.reasyn_viz import parse_reasyn_results, display_synthesis_table

    synthesis_df = parse_reasyn_results(reasyn_results, compound_ids)
    display_synthesis_table(synthesis_df, total_submitted=len(smiles_list))
else:
    synthesis_df = pd.DataFrame()
    print("No ReaSyn results to display.")


In [ ]:
if reasyn_results:
    from cdk_oracle.reasyn_viz import print_pathway_details

    print_pathway_details(reasyn_results, compound_ids, top_n=1)
else:
    print("No ReaSyn pathways to inspect.")


In [ ]:
if reasyn_results and not synthesis_df.empty:
    synthesis_file = config.output_dir / "synthesis_pathways.csv"
    synthesis_df.to_csv(synthesis_file, index=False)
    print(f"Synthesis results saved to: {synthesis_file}")

    with open(config.output_dir / "reasyn_full_results.json", "w") as f:
        json.dump(reasyn_results, f, indent=2)
    print(f"Full ReaSyn JSON saved to:  {config.output_dir / 'reasyn_full_results.json'}")
else:
    print("Skipping ReaSyn output files.")


---

## Alternative: Evaluate Existing Compounds

If you have your own SMILES to evaluate (skip generation):

In [ ]:
# Clean output
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Example: Evaluate custom compounds
pipeline = CDKDesignPipeline(config, boltz2_endpoints=BOLTZ2_ENDPOINTS)


my_compounds = [
    "CC1=CC=C(C=C1)NC2=NC=NC3=C2C=C(C=C3)F",
    "CN1CCN(CC1)C2=CC=C(C=C2)NC3=NC=NC4=C3C=CC=C4",
]

eval_results = pipeline.evaluate_existing(
    smiles_list=my_compounds,
    use_msa=True,
    verbose=True
)
display(eval_results.scores_df)

---

## Quick Reference

### Scoring Weights
| Component | Weight | Description |
|-----------|--------|-------------|
| binding_affinity | 0.25 | CDK4 potency (lower IC50 = better) |
| selectivity | 0.20 | CDK11/CDK4 ratio (higher = better) |
| cdk11_avoidance | 0.15 | Avoiding CDK11 (higher IC50 = better) |
| qed | 0.15 | Drug-likeness (0-1) |
| sa | 0.10 | Synthetic accessibility |
| pains | 0.10 | PAINS-free (binary) |
| novelty | 0.05 | Structural novelty |

### Desirable Properties
- **CDK4 IC50**: < 100 nM (ideally < 10 nM)
- **CDK11 IC50**: > 1,000 nM (ideally > 10,000 nM)
- **Selectivity**: > 10x (ideally > 100x)
- **QED**: > 0.5
- **Lipinski violations**: ≤ 1
- **PAINS alerts**: 0